### importing packages and loading data

In [1]:
%matplotlib notebook

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("C:/Users/moomo/OneDrive/Desktop/cisco stock data kaggle.csv")
print("Shape:", df.shape)

Shape: (8823, 6)


# cleaning the data

### missing values

In [3]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
datetime          0
open              0
high              0
low               0
close             0
percent_change    0
dtype: int64


### converting data colunm 

In [4]:
df["datetime"] = pd.to_datetime(df["datetime"])


### remove duplicates 

In [5]:
df = df.drop_duplicates()

### sorting by data 

In [6]:
df = df.sort_values("datetime")


### validating price data

In [7]:
invalid_rows = df[
    (df["high"] < df["open"]) |
    (df["high"] < df["close"]) |
    (df["high"] < df["low"]) |
    (df["low"] > df["open"]) |
    (df["low"] > df["close"])
]

print("\nInvalid Rows Found:", len(invalid_rows))


Invalid Rows Found: 0


### removing invalid rows 

In [8]:
df = df.drop(invalid_rows.index)

### printing columns 

In [9]:
price_cols = ["open", "high", "low", "close"]
for col in price_cols:
    df = df[df[col] > 0]

### daily return

In [10]:
df["daily_return_pct"] = (
    (df["close"] - df["open"]) / df["open"]
) * 100

### checking updated columns if any

In [11]:
print(df.head())

       datetime  open  high   low  close  percent_change  daily_return_pct
8822 1990-02-20  0.08  0.08  0.08   0.08             0.0               0.0
8821 1990-02-21  0.08  0.08  0.08   0.08             0.0               0.0
8820 1990-02-22  0.08  0.08  0.08   0.08             0.0               0.0
8819 1990-02-23  0.08  0.08  0.08   0.08             0.0               0.0
8818 1990-02-26  0.08  0.08  0.08   0.08             0.0               0.0


### review of all changes 

In [12]:
print("\nFinal Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)

print("\nSummary Statistics:")
print(df.describe())


Final Shape: (8823, 7)

Data Types:
datetime            datetime64[ns]
open                       float64
high                       float64
low                        float64
close                      float64
percent_change             float64
daily_return_pct           float64
dtype: object

Summary Statistics:
                            datetime         open         high          low  \
count                           8823  8823.000000  8823.000000  8823.000000   
mean   2007-08-22 05:08:18.197892096    24.558217    24.849671    24.254359   
min              1990-02-20 00:00:00     0.070000     0.070000     0.070000   
25%              1998-11-11 12:00:00    13.240000    13.530000    13.010000   
50%              2007-08-22 00:00:00    22.000000    22.260000    21.720000   
75%              2016-05-25 12:00:00    34.230000    34.480000    33.890000   
max              2025-03-07 00:00:00    81.440000    82.000000    79.060000   
std                              NaN    17.359392  

# statistical analysis of data

### moving averages 

In [13]:
df["MA20"] = df["close"].rolling(20).mean()
df["MA50"] = df["close"].rolling(50).mean()
df["MA200"] = df["close"].rolling(200).mean()

### descriptive statistics

In [14]:
print("\nSummary Statistics")
print(df.describe())

print("\nHighest Close")
print(df.loc[df["close"].idxmax()])

print("\nLowest Close")
print(df.loc[df["close"].idxmin()])


Summary Statistics
                            datetime         open         high          low  \
count                           8823  8823.000000  8823.000000  8823.000000   
mean   2007-08-22 05:08:18.197892096    24.558217    24.849671    24.254359   
min              1990-02-20 00:00:00     0.070000     0.070000     0.070000   
25%              1998-11-11 12:00:00    13.240000    13.530000    13.010000   
50%              2007-08-22 00:00:00    22.000000    22.260000    21.720000   
75%              2016-05-25 12:00:00    34.230000    34.480000    33.890000   
max              2025-03-07 00:00:00    81.440000    82.000000    79.060000   
std                              NaN    17.359392    17.543504    17.163657   

             close  percent_change  daily_return_pct         MA20  \
count  8823.000000     8823.000000       8823.000000  8804.000000   
mean     24.557148        0.034902          0.034902    24.541186   
min       0.070000      -19.117647        -19.117647     0.08

In [15]:
df["daily_return"] = df["close"].pct_change()


annual_return = (
    df["daily_return"].mean() * 252
)
print('annual return is: ', annual_return)

annual_volatility = (
    df["daily_return"].std() * np.sqrt(252)
)
print('annual volatility is: ', annual_volatility)

sharpe_ratio = (
    annual_return / annual_volatility
)
print('sharpe ratio is : ', sharpe_ratio)


annual return is:  0.2790922031916191
annual volatility is:  0.4205186286673729
sharpe ratio is :  0.6636857065668283


### correlation matrix 

In [16]:
corr = df[
    ["open","high","low","close"]
].corr()

print("\nCorrelation Matrix")
print(corr)


Correlation Matrix
           open      high       low     close
open   1.000000  0.999720  0.999575  0.999484
high   0.999720  1.000000  0.999288  0.999723
low    0.999575  0.999288  1.000000  0.999529
close  0.999484  0.999723  0.999529  1.000000
